# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata as an object, not a dictionary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

### List all Record Sets
We enumerate all record sets present in the dataset using their `@id` fields. For each record set, we also display its fields and available columns.

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.id} ({field.name})")
    if hasattr(rs, 'columns') and rs.columns:
        print(f"  Columns:")
        for col in rs.columns:
            print(f"    - {col.id} ({col.name})")
    print('-' * 60)

### Explore Records from a Record Set
Let's preview some actual records from one of the record sets — use the `@id` field.


In [ ]:
# Pick the first record set for preview
if len(record_sets) == 0:
    raise Exception('No record sets found in this dataset.')

first_record_set = record_sets[0]
print(f"Previewing records from record set @id: {first_record_set.id}\n")
for i, record in enumerate(dataset.records(record_set=first_record_set.id)):
    pprint(record)
    if i >= 2:
        break

## 3. Data Extraction
Load records from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = dict()
record_set_ids = [rs.id for rs in record_sets]
print(f"Record sets found: {record_set_ids}\n")
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        print(f"No records loaded for record set {rs_id}")
# For demonstration, use the first loaded DataFrame
main_rs_id = None
for rs_id in dataframes:
    main_rs_id = rs_id
    break

if main_rs_id is not None:
    print(f"\nColumns in {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print('No dataframes with records available.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by attributes. All columns are referenced by their unique `@id`.

### Example: Filtering and Normalizing a Numeric Field


In [ ]:
from pandas.api.types import is_numeric_dtype

# Choose the primary dataframe and identify a numeric field by @id
df = dataframes[main_rs_id]

# Automatically select a numeric field (column) by checking datatypes
numeric_field_id = None
for col in df.columns:
    if is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print('No numeric fields found in this record set.')
else:
    print(f"Using numeric field @id: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() # Example threshold: mean value
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}, shape: {filtered_df.shape}")

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Select a group field (non-numeric, for grouping)
    group_field_id = None
    for col in df.columns:
        if not is_numeric_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id:
        print(f"\nGrouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print('No group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the numeric field and, if available, a group comparison.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if 'numeric_field_id' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # If grouping is available
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using the Croissant schema and `mlcroissant`. We demonstrated how to:

- Load metadata and review record sets via their `@id`
- Extract tabular data, referencing all fields by their `@id`
- Perform basic data analyses such as filtering, normalization, and grouping
- Visualize key numeric field distributions and group comparisons

All data references were made using `@id` as required for reproducible and schema-driven exploration.